# OI Signal Engine — data export and batch backtest

Run this notebook in Google Colab from a network where Binance USD-M public market data is legally available. It uses public endpoints only; no API keys are needed. It does not bypass HTTP 451 or regional restrictions.

It exports recent 15m candles and OI for BTC, ETH, SOL, BNB and XRP, runs the repository's existing backtest unchanged, and packages the CSVs and JSON report into one ZIP. Binance OI history is capped at 500 samples (about five days), so this is only a smoke test, not robust validation.

In [ ]:
!git clone --depth 1 --branch oi-signal-engine https://github.com/aliaminiaa68aa-alt/Pump-hunter-bot.git
%cd Pump-hunter-bot
!python -m pip -q install requests

In [ ]:
from pathlib import Path
import json, zipfile
from oi_engine.export_history import export_symbols
from oi_engine.batch_backtest import run_batch_backtest

symbols = ["BTCUSDT", "ETHUSDT", "SOLUSDT", "BNBUSDT", "XRPUSDT"]
data_dir = Path("oi-data")
data_dir.mkdir(exist_ok=True)

export_results = export_symbols(symbols, str(data_dir))
print("EXPORT RESULTS")
print(json.dumps(export_results, indent=2))

report = run_batch_backtest(symbols, str(data_dir))
report_path = data_dir / "batch_backtest_report.json"
report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")

print("\nBACKTEST SUMMARY")
print(json.dumps({
    "symbols_succeeded": report["symbols_succeeded"],
    "symbols_failed": report["symbols_failed"],
    "pooled_trade_count": report["pooled_trade_count"],
    "per_symbol_metrics": {
        symbol: details["metrics"] for symbol, details in report["per_symbol"].items()
    },
}, ensure_ascii=False, indent=2))

In [ ]:
zip_path = Path("oi_backtest_bundle.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    if data_dir.exists():
        for path in data_dir.glob("*"):
            if path.is_file():
                archive.write(path, arcname=f"oi-data/{path.name}")
    archive.write(report_path, arcname="oi-data/batch_backtest_report.json")

print(f"Created {zip_path} — upload this ZIP back in the ChatGPT conversation for review.")
try:
    from google.colab import files
    files.download(str(zip_path))
except ImportError:
    print("Not running in Colab; download oi_backtest_bundle.zip from the notebook file browser.")